In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Input,GlobalAveragePooling2D, Dense,Dropout,BatchNormalization
from tensorflow.keras.models import Model
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.callbacks import EarlyStopping

img_height,img_width=224,224
num_class=2
num_folds=5
epochs=50
batch_size=32

In [6]:
import numpy as np
import tensorflow as tf

# Define parameters
img_height, img_width = 224, 224  # Set your desired image height and width
batch_size = 32  # Set your desired batch size

# Load training dataset with the correct image size
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    '/kaggle/input/aug2splitpcos/splitAug2PreprocessPcos2000/train',
    image_size=(img_height, img_width),  # Ensure images are resized to 224x224
    batch_size=batch_size
)

# Load testing dataset with the correct image size
test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    '/kaggle/input/aug2splitpcos/splitAug2PreprocessPcos2000/val',
    image_size=(img_height, img_width),  # Ensure images are resized to 224x224
    batch_size=batch_size
)

# Extracting x_train, y_train
x_train = []
y_train = []
for images, labels in train_dataset:
    x_train.extend(images.numpy())
    y_train.extend(labels.numpy())

# Extracting x_test, y_test
x_test = []
y_test = []
for images, labels in test_dataset:
    x_test.extend(images.numpy())
    y_test.extend(labels.numpy())

# Convert lists to numpy arrays
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}, y_test shape: {y_test.shape}")


Found 3200 files belonging to 2 classes.
Found 800 files belonging to 2 classes.
x_train shape: (3200, 224, 224, 3), y_train shape: (3200,)
x_test shape: (800, 224, 224, 3), y_test shape: (800,)


In [7]:
skf=StratifiedKFold(n_splits=num_folds, shuffle=True,random_state=42)
fold_accuracies=[]
num_classes=2

In [8]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.callbacks import EarlyStopping

# Define parameters
img_height, img_width = 224, 224
num_classes = len(set(y_train))  # Get the number of classes dynamically
num_folds = 5
epochs = 10  # Increased epochs for better training
batch_size = 32

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

# Store accuracy for each fold
fold_accuracies = []

for train_index, val_index in skf.split(x_train, y_train):
    x_train_fold, x_val_fold = x_train[train_index], x_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    # Define the base model (DenseNet121)
    # Base model
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
    input_layer = Input(shape=(img_height, img_width, 3))
    base_model_output = base_model(input_layer, training=False)

# Feature extraction
    x = Conv2D(filters=64, kernel_size=(3,3), activation='relu', padding='same')(base_model_output)
    x = GlobalAveragePooling2D()(x)

# Fully connected layers
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.7)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.2)(x)

# Output layer
    output_layer = Dense(1, activation='sigmoid')(x)

# Create and compile the model
    model = Model(inputs=input_layer, outputs=output_layer)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    # Early stopping callback
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    # Train the model using the current fold
    history = model.fit(x_train_fold, y_train_fold,
                        validation_data=(x_val_fold, y_val_fold),
                        epochs=epochs,
                        batch_size=batch_size,
                        verbose=1,
                        callbacks=[early_stopping])  # Added early stopping

    # Evaluate the model on the validation set
    val_loss, val_acc = model.evaluate(x_val_fold, y_val_fold)
    fold_accuracies.append(val_acc)
    print(f"Fold Validation Accuracy: {val_acc:.4f}")

# Calculate and print the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"Average Validation Accuracy across {num_folds} folds: {average_accuracy:.4f}")

Epoch 1/10


I0000 00:00:1750060894.400229     102 service.cc:148] XLA service 0x79be50003810 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750060894.400975     102 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1750060894.401000     102 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1750060899.689417     102 cuda_dnn.cc:529] Loaded cuDNN version 90300
E0000 00:00:1750060910.559901     102 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1750060910.702443     102 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1750060911.163805     102 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. Th

80/80 ━━━━━━━━━━━━━━━━━━━━ 114s 237ms/step - accuracy: 0.7363 - loss: 0.4921 - val_accuracy: 0.9656 - val_loss: 0.2019
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 137ms/step - accuracy: 0.9977 - loss: 0.0199 - val_accuracy: 0.9891 - val_loss: 0.0700
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 139ms/step - accuracy: 0.9994 - loss: 0.0079 - val_accuracy: 1.0000 - val_loss: 0.0102
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 140ms/step - accuracy: 0.9939 - loss: 0.0202 - val_accuracy: 0.9953 - val_loss: 0.0161
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 142ms/step - accuracy: 0.9969 - loss: 0.0139 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 140ms/step - accuracy: 0.9997 - loss: 0.0028 - val_accuracy: 1.0000 - val_loss: 3.5237e-04
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 137ms/step - accuracy: 0.9998 - loss: 0.0015 - val_accuracy: 1.0000 - val_loss: 3.6060e-04
Epoch 8/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 137ms/step - accuracy: 0.9987 - loss: 0.0034 - val_accura

In [9]:
model.save('/kaggle/working/ImportEfficientPcos(New).h5')